# Music Streaming Behavior Analysis — Springfield vs Shelbyville

**Hypothesis:** User activity on the streaming platform differs depending on the day of the week and the city.

**Dataset:** 65,079 streaming records from an online music service, containing track plays, genres, cities, and timestamps.

## 1. Setup & Data Loading

In [ ]:
import pandas as pd

df = pd.read_csv('/datasets/music_project_en.csv')
print(f'Shape: {df.shape}')
df.head()

## 2. Data Preprocessing

### 2.1 Column Name Standardization

In [ ]:
# Lowercase, strip whitespace, apply snake_case
df.columns = [col.lower().strip() for col in df.columns]
df = df.rename(columns={'userid': 'user_id'})

print('Columns:', df.columns.tolist())

### 2.2 Missing Values

In [ ]:
print('Missing values per column:')
print(df.isna().sum())

# Fill missing track, artist, genre with 'unknown'
for col in ['track', 'artist', 'genre']:
    df[col] = df[col].fillna('unknown')

print(f'\nTotal missing values after fill: {df.isna().sum().sum()}')

**Note:** Missing values in `track` and `artist` are non-critical and filled with `'unknown'`. Missing `genre` values (1,198 rows) could affect genre-based analysis but are unavoidable without the original data source.

### 2.3 Duplicate Removal

In [ ]:
print(f'Explicit duplicates found: {df.duplicated().sum()}')
df = df.drop_duplicates()
print(f'Duplicates after removal: {df.duplicated().sum()}')
print(f'Remaining records: {len(df):,}')

### 2.4 Implicit Genre Duplicates

In [ ]:
def replace_wrong_genres(wrong_genres, correct_genre):
    """Standardize genre variants to a single canonical label."""
    for wrong in wrong_genres:
        df.loc[df['genre'] == wrong, 'genre'] = correct_genre

# Consolidate hiphop variants
replace_wrong_genres(['hip', 'hop', 'hip-hop', 'hip hop'], 'hiphop')

# Verify no variants remain
variants = ['hip', 'hop', 'hip-hop', 'hip hop']
remaining = df[df['genre'].isin(variants)]['genre'].value_counts()
print('Remaining hiphop variants:', remaining.to_dict() if len(remaining) else 'None ✅')

## 3. Hypothesis Testing — User Activity by City and Day

**H₀:** User activity (track plays) is equal across cities regardless of day of the week.

**H₁:** User activity differs between Springfield and Shelbyville depending on the day.

In [ ]:
def number_tracks(day, city):
    """Count track plays for a given day and city."""
    return df[(df['day'] == day) & (df['city'] == city)]['user_id'].count()


days  = ['Monday', 'Wednesday', 'Friday']
cities = ['Springfield', 'Shelbyville']

results = pd.DataFrame({
    'Day': days,
    'Springfield': [number_tracks(d, 'Springfield') for d in days],
    'Shelbyville': [number_tracks(d, 'Shelbyville') for d in days]
})
results['Difference'] = results['Springfield'] - results['Shelbyville']
results['Springfield Advantage'] = (results['Difference'] / results['Shelbyville'] * 100).round(1).astype(str) + '%'
results

In [ ]:
import matplotlib.pyplot as plt

x = range(len(days))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar([i - width/2 for i in x], results['Springfield'], width, label='Springfield', color='steelblue')
ax.bar([i + width/2 for i in x], results['Shelbyville'], width, label='Shelbyville', color='darkorange')

ax.set_title('Track Plays by City and Day of Week')
ax.set_xlabel('Day')
ax.set_ylabel('Number of Plays')
ax.set_xticks(list(x))
ax.set_xticklabels(days)
ax.legend()
plt.tight_layout()
plt.show()

## 4. Conclusion

| Day | Springfield | Shelbyville | Difference |
|---|---|---|---|
| Monday | 15,740 | 5,614 | +10,126 |
| Wednesday | 11,056 | 7,003 | +4,053 |
| Friday | 15,945 | 5,895 | +10,050 |

**The hypothesis is confirmed** — Springfield consistently shows higher playback counts across all three days, with the gap narrowing on Wednesdays.

**Important caveat:** This analysis uses raw play counts without normalizing by city population. Springfield's higher numbers may partly reflect a larger user base rather than higher per-capita engagement. A per-user activity metric would provide a more accurate behavioral comparison.